# Job Queue with Status Page Example

This notebook demonstrates a job queue system with a comprehensive status page and job management features.

In [1]:
from fasthtml.common import *
import uuid, time, threading
from datetime import datetime
from typing import Dict, Any

from cjm_tqdm_capture.progress_monitor import ProgressMonitor
from cjm_tqdm_capture.job_runner import JobRunner
from cjm_tqdm_capture.streaming import sse_stream

In [2]:
# For Jupyter display
from fasthtml.jupyter import JupyUvi, HTMX
from cjm_fasthtml_daisyui.core.testing import create_test_app, create_test_page, start_test_server
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from IPython.display import display

# Import DaisyUI factories
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors, btn_sizes, btn_behaviors
from cjm_fasthtml_daisyui.components.actions.modal import modal, modal_box, modal_action
from cjm_fasthtml_daisyui.components.feedback.progress import progress, progress_colors
from cjm_fasthtml_daisyui.components.feedback.alert import alert, alert_colors
from cjm_fasthtml_daisyui.components.data_display.card import card
from cjm_fasthtml_daisyui.components.data_display.table import table, table_modifiers
from cjm_fasthtml_daisyui.components.data_display.badge import badge, badge_colors
from cjm_fasthtml_daisyui.components.data_display.stat import stat, stat_title, stat_value, stats
from cjm_fasthtml_daisyui.components.data_input.text_input import text_input
from cjm_fasthtml_daisyui.components.data_input.select import select
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, text_dui

# Import Tailwind factories
from cjm_fasthtml_tailwind.utilities.spacing import p, m, space
from cjm_fasthtml_tailwind.utilities.typography import font_weight, font_size, text_color
from cjm_fasthtml_tailwind.utilities.sizing import w, max_w, max_h, container
from cjm_fasthtml_tailwind.utilities.layout import overflow
from cjm_fasthtml_tailwind.utilities.borders import rounded
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import flex_display, gap
from cjm_fasthtml_tailwind.utilities.effects import shadow
from cjm_fasthtml_tailwind.core.base import combine_classes

In [3]:
# Create app
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from cjm_fasthtml_daisyui.core.testing import create_test_app

app, rt = create_test_app(theme=DaisyUITheme.BUSINESS)

# Initialize with history for job tracking
monitor = ProgressMonitor(keep_history=True, history_limit=200)
runner = JobRunner(monitor)

# Job metadata storage
job_metadata: Dict[str, Any] = {}
job_results: Dict[str, Any] = {}

# Track active jobs and button state
active_jobs = set()
last_button_state = False  # Track last known button disabled state

def render_submit_button(disabled=False):
    """Render submit button with appropriate state"""
    btn_classes = combine_classes(
        btn,
        btn_colors.primary,
        btn_behaviors.disabled if disabled else ""
    )
    
    return Button(
        "Submit Job",
        type="submit",
        id="submit-job-btn",
        cls=btn_classes,
        disabled=disabled,
        hx_swap_oob="true"  # Enable out-of-band swap
    )

In [4]:
# Cancellable job runner extension
class CancellableJobRunner(JobRunner):
    def __init__(self, monitor):
        super().__init__(monitor)
        self._stop_events = {}
    
    def start_cancellable(self, job_id, fn, *args, patch_kwargs=None, **kwargs):
        stop_event = threading.Event()
        self._stop_events[job_id] = stop_event
        
        def wrapper():
            try:
                result = fn(stop_event, *args, **kwargs)
                job_results[job_id] = {"status": "success", "data": result}
            except Exception as e:
                job_results[job_id] = {"status": "error", "error": str(e)}
            finally:
                # Clean up stop event when job finishes
                if job_id in self._stop_events:
                    del self._stop_events[job_id]
        
        return self.start(job_id, wrapper, patch_kwargs=patch_kwargs)
    
    def cancel(self, job_id):
        if job_id in self._stop_events:
            # Set the stop event to signal the job to stop
            self._stop_events[job_id].set()
            
            # Mark job as cancelled in metadata
            if job_id in job_metadata:
                job_metadata[job_id]["status"] = "cancelled"
            
            # Mark job as cancelled in results
            job_results[job_id] = {"status": "cancelled"}
            
            # Force mark as completed in monitor so it can be cleared
            # This is a workaround since the monitor doesn't have a native cancel method
            snapshot = monitor.snapshot(job_id)
            if snapshot:
                # Update the monitor's internal state to mark as completed
                # This ensures the job shows as completed and can be cleared
                monitor._jobs[job_id]['completed'] = True
                monitor._jobs[job_id]['overall_progress'] = snapshot['overall_progress']
            
            return True
        return False

# Use cancellable runner
runner = CancellableJobRunner(monitor)

In [5]:
# Various job types
def batch_processing_job(stop_event, batch_size=1000, delay=0.01):
    from tqdm import tqdm
    import time
    
    results = []
    
    # Process in batches
    for batch in range(3):
        desc = f"Batch {batch + 1}/3"
        for i in tqdm(range(batch_size), desc=desc):
            if stop_event.is_set():
                return {"status": "cancelled", "processed": results}
            time.sleep(delay)
            results.append(f"item_{batch}_{i}")
    
    return {"status": "complete", "processed": results}

def data_export_job(stop_event, format_type="csv", records=200):
    from tqdm import tqdm
    import time
    
    # Fetch data
    for _ in tqdm(range(records // 2), desc="Fetching data"):
        if stop_event.is_set():
            return {"status": "cancelled"}
        time.sleep(0.01)
    
    # Format data
    for _ in tqdm(range(records // 4), desc=f"Formatting as {format_type}"):
        if stop_event.is_set():
            return {"status": "cancelled"}
        time.sleep(0.02)
    
    # Write to file
    for _ in tqdm(range(records // 4), desc="Writing to file"):
        if stop_event.is_set():
            return {"status": "cancelled"}
        time.sleep(0.01)
    
    return {"status": "complete", "file": f"export_{format_type}_{records}.{format_type}"}

def model_training_job(stop_event, epochs=10, batch_size=32):
    from tqdm import tqdm
    import time
    import random
    
    metrics = []
    
    for epoch in range(epochs):
        # Training
        for _ in tqdm(range(1000), desc=f"Epoch {epoch+1}/{epochs} - Training"):
            if stop_event.is_set():
                return {"status": "cancelled", "metrics": metrics}
            time.sleep(0.005)
        
        # Validation
        for _ in tqdm(range(200), desc=f"Epoch {epoch+1}/{epochs} - Validation"):
            if stop_event.is_set():
                return {"status": "cancelled", "metrics": metrics}
            time.sleep(0.01)
        
        # Record metrics
        metrics.append({
            "epoch": epoch + 1,
            "loss": random.uniform(0.1, 1.0),
            "accuracy": random.uniform(0.7, 0.99)
        })
    
    return {"status": "complete", "metrics": metrics}

In [6]:
# Job queue management page with HTMX
from cjm_fasthtml_daisyui.core.testing import create_test_page

@rt("/")
def index():
    return create_test_page(
        "Job Queue Manager",
        Div(
            H1("Job Queue Management System", cls=combine_classes(font_size._3xl, font_weight.bold, m.b(6))),
            
            # Job creation panel
            Div(
                H2("Create New Job", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
                Form(
                    Select(
                        Option("Batch Processing", value="batch"),
                        Option("Data Export", value="export"),
                        Option("Model Training", value="training"),
                        name="type",
                        cls=combine_classes(select, w.full, m.b(3))
                    ),
                    Input(
                        name="name",
                        type="text",
                        placeholder="Job name (optional)",
                        cls=combine_classes(text_input, w.full, m.b(3))
                    ),
                    Div(
                        Button(
                            "Submit Job",
                            type="submit",
                            id="submit-job-btn",
                            cls=combine_classes(btn, btn_colors.primary)
                        ),
                        Button(
                            "Clear Finished",
                            title="Clear completed and cancelled jobs",
                            hx_post="/clear_completed",
                            hx_target="#job-queue",
                            hx_swap="innerHTML",
                            cls=combine_classes(btn, btn_colors.warning, m.l(2))
                        ),
                        cls=combine_classes(flex_display, gap(2))
                    ),
                    hx_post="/create_job",
                    hx_target="#job-queue",
                    hx_swap="innerHTML",
                    # Also trigger stats update after job creation
                    hx_trigger_after_swap="htmx:afterSwap from:body",
                    hx_on_after_request="this.reset()",
                ),
                cls=combine_classes(card, bg_dui.base_200, p(6), m.b(6))
            ),
            
            # Statistics that auto-refresh only when jobs are running
            Div(
                H2("Queue Statistics", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
                Div(
                    hx_get="/stats",
                    hx_trigger="load",  # Only load initially, polling added conditionally
                    hx_swap="outerHTML",  # Use outerHTML to replace the entire element
                    id="stats",
                    cls=combine_classes(stats, shadow(), w.full)
                ),
                cls=str(m.b(6))
            ),
            
            # Job queue table - polling added conditionally
            Div(
                H2("Job Queue", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
                Div(
                    hx_get="/queue",
                    hx_trigger="load",  # Only load initially, polling added conditionally
                    hx_swap="innerHTML",
                    id="job-queue",
                    cls=str(overflow.x.auto)
                ),
                cls=combine_classes(card, bg_dui.base_200, p(6), m.b(6))
            ),
            
            # Job details modal
            Dialog(
                Div(
                    H3("Job Details", cls=combine_classes(font_weight.bold, font_size.lg, m.b(4))),
                    Div(id="job-details-content"),
                    Div(
                        Button(
                            "Close",
                            onclick="this.closest('dialog').close()",
                            cls=combine_classes(btn, btn_sizes.sm)
                        ),
                        cls=str(modal_action)
                    ),
                    cls=combine_classes(modal_box, max_w._4xl)
                ),
                id="job-modal",
                cls=str(modal)
            ),
            
            cls=combine_classes(container, m.x.auto, p(8))
        )
    )

In [7]:
# API endpoints using FastHTML conventions
@rt("/create_job")
async def create_job(request):
    global last_button_state
    print(f"[CREATE_JOB] Starting job creation")
    
    form = await request.form()
    job_type = form.get('type', 'batch')
    job_name = form.get('name', '')
    
    job_id = str(uuid.uuid4())
    active_jobs.add(job_id)  # Track active job
    print(f"[CREATE_JOB] Created job {job_id[:8]}... type={job_type}")
    
    # Select job function with appropriate parameters
    job_configs = {
        'batch': (batch_processing_job, {'batch_size': 50, 'delay': 0.005}),
        'export': (data_export_job, {'format_type': 'csv', 'records': 100}),
        'training': (model_training_job, {'epochs': 3, 'batch_size': 32})
    }
    
    job_fn, kwargs = job_configs.get(job_type, (batch_processing_job, {}))
    
    # Store metadata
    job_metadata[job_id] = {
        'id': job_id,
        'name': job_name or f"{job_type.title()} Job",
        'type': job_type,
        'status': 'running',
        'created_at': datetime.now().isoformat(),
        'params': kwargs
    }
    
    # Start job with appropriate throttling
    runner.start_cancellable(
        job_id,
        job_fn,
        **kwargs,
        patch_kwargs={
            "min_delta_pct": 5,
            "min_update_interval": 0.05,
            "emit_initial": True
        }
    )
    
    # Small delay to allow monitor to register the job
    import time
    time.sleep(0.1)  # 100ms delay
    print(f"[CREATE_JOB] After delay, checking monitor...")
    
    # Force button state update since we're starting a new job
    print(f"[CREATE_JOB] Setting last_button_state from {last_button_state} to True")
    last_button_state = True
    
    # Get queue content (this will include polling triggers)
    print(f"[CREATE_JOB] Calling queue() to get content")
    queue_content = queue()
    print(f"[CREATE_JOB] Queue content type: {type(queue_content)}")
    print(f"[CREATE_JOB] Queue content attrs: {getattr(queue_content, 'attrs', {})}")
    
    # Always include button update in create_job response (force disabled state)
    button_update = render_submit_button(disabled=True)
    print(f"[CREATE_JOB] Forcing button disabled state")
    
    # Add stats OOB update
    print(f"[CREATE_JOB] Calling stats() to get content")
    stats_update = stats()
    stats_update.attrs['id'] = 'stats'
    stats_update.attrs['hx-swap-oob'] = 'true'
    print(f"[CREATE_JOB] Stats attrs: {stats_update.attrs}")
    
    # Return combined response
    response = Div(
        queue_content,  # Main content for #job-queue
        button_update,  # Force button to disabled state
        stats_update    # OOB update for stats
    )
    print(f"[CREATE_JOB] Returning response with {len(response.children)} children")
    return response

In [8]:
@rt("/queue")
def queue():
    global last_button_state
    print(f"[QUEUE] Called, last_button_state={last_button_state}")
    
    jobs = monitor.all()
    print(f"[QUEUE] Found {len(jobs)} jobs")
    
    # Check if any jobs are running
    has_running = False
    
    if not jobs:
        # No jobs - check if button state changed
        button_update = None
        if last_button_state != False:
            print(f"[QUEUE] No jobs, updating button state from {last_button_state} to False")
            last_button_state = False
            button_update = render_submit_button(disabled=False)
        
        result = P("No jobs in queue", cls=str(text_color.gray(500)))
        
        # Include button update only if state changed
        if button_update:
            return Div(button_update, result)
        return result
    
    rows = []
    for job_id, job in jobs.items():
        meta = job_metadata.get(job_id, {})
        
        # Determine status - check metadata first for cancelled status
        if meta.get('status') == 'cancelled':
            status_text = "Cancelled"
            status_color = badge_colors.error
        elif job['completed']:
            status_text = "Complete"
            status_color = badge_colors.success
        else:
            status_text = "Running"
            status_color = badge_colors.info
            has_running = True  # Found a running job
        
        print(f"[QUEUE] Job {job_id[:8]}... status={status_text}, progress={job['overall_progress']:.1f}%")
        
        rows.append(
            Tr(
                Td(job_id[:8] + "..."),
                Td(meta.get('name', 'Unknown')),
                Td(meta.get('type', 'unknown').title()),
                Td(
                    Progress(
                        value=str(job['overall_progress']),
                        max="100",
                        cls=combine_classes(progress, progress_colors.primary, w(20))
                    )
                ),
                Td(
                    Span(
                        status_text,
                        cls=combine_classes(badge, status_color)
                    )
                ),
                Td(
                    Button(
                        "View",
                        hx_get=f"/job_details?job_id={job_id}",
                        hx_target="#job-details-content",
                        hx_swap="innerHTML",
                        onclick="document.getElementById('job-modal').showModal()",
                        cls=combine_classes(btn, btn_sizes.xs, btn_colors.primary)
                    ),
                    Button(
                        "Cancel",
                        hx_post=f"/cancel_job?job_id={job_id}",
                        hx_target="#job-queue",
                        hx_swap="innerHTML",
                        cls=combine_classes(btn, btn_sizes.xs, btn_colors.error, m.l(1))
                    ) if not job['completed'] and meta.get('status') != 'cancelled' else ""
                )
            )
        )
    
    print(f"[QUEUE] has_running={has_running}")
    
    # Check if button state changed
    button_update = None
    if last_button_state != has_running:
        print(f"[QUEUE] Button state changed from {last_button_state} to {has_running}")
        last_button_state = has_running
        button_update = render_submit_button(disabled=has_running)
    
    # Build the table
    table_content = Table(
        Thead(
            Tr(
                Th("ID"),
                Th("Name"),
                Th("Type"),
                Th("Progress"),
                Th("Status"),
                Th("Actions")
            )
        ),
        Tbody(*rows),
        cls=combine_classes(table, table_modifiers.zebra, w.full)
    )
    
    # Create wrapper div that will contain the table and polling info
    wrapper = Div(table_content)
    
    # Add polling trigger if jobs are running - poll the wrapper itself
    if has_running:
        print(f"[QUEUE] Adding polling triggers to wrapper")
        wrapper.attrs['hx-get'] = "/queue"
        wrapper.attrs['hx-trigger'] = "load delay:1s"
        wrapper.attrs['hx-swap'] = "innerHTML"
        wrapper.attrs['hx-target'] = "#job-queue"
        # Also trigger stats refresh via JavaScript
        wrapper.attrs['hx-on-htmx-after-swap'] = "htmx.trigger('#stats', 'refresh')"
    else:
        print(f"[QUEUE] No polling needed - all jobs complete")
    
    # Return wrapper with button update only if state changed
    if button_update:
        print(f"[QUEUE] Returning wrapper with button update")
        return Div(button_update, wrapper)
    
    print(f"[QUEUE] Returning wrapper without button update")
    return wrapper

In [9]:
@rt("/stats")
def stats():
    print(f"[STATS] Called")
    jobs = monitor.all()
    
    total = len(jobs)
    running = 0
    completed = 0
    cancelled = 0
    
    for job_id, job in jobs.items():
        meta = job_metadata.get(job_id, {})
        if meta.get('status') == 'cancelled':
            cancelled += 1
        elif job['completed']:
            completed += 1
        else:
            running += 1
    
    print(f"[STATS] Total={total}, Running={running}, Completed={completed}, Cancelled={cancelled}")
    
    stats_content = Div(
        Div(
            Div("Total Jobs", cls=str(stat_title)),
            Div(str(total), cls=str(stat_value)),
            cls=str(stat)
        ),
        Div(
            Div("Running", cls=str(stat_title)),
            Div(str(running), cls=combine_classes(stat_value, text_dui.primary)),
            cls=str(stat)
        ),
        Div(
            Div("Completed", cls=str(stat_title)),
            Div(str(completed), cls=combine_classes(stat_value, text_dui.success)),
            cls=str(stat)
        ),
        Div(
            Div("Cancelled", cls=str(stat_title)),
            Div(str(cancelled), cls=combine_classes(stat_value, text_dui.error)),
            cls=str(stat)
        ),
        id="stats",  # Always include the ID
        cls=combine_classes(stats, shadow(), w.full)
    )
    
    # Add polling trigger only if jobs are running
    if running > 0:
        print(f"[STATS] Adding polling triggers (running jobs found)")
        stats_content.attrs['hx-get'] = "/stats"
        stats_content.attrs['hx-trigger'] = "load delay:2s"
        stats_content.attrs['hx-swap'] = "outerHTML"
    else:
        print(f"[STATS] No polling needed - no running jobs")
    
    print(f"[STATS] Returning with attrs: {stats_content.attrs}")
    return stats_content

In [10]:
@rt("/job_details")
def job_details(job_id: str):
    """Job details with auto-refresh while running"""
    import json
    snapshot = monitor.snapshot(job_id)
    meta = job_metadata.get(job_id, {})
    result = job_results.get(job_id)
    
    if not snapshot:
        return Div("Job not found", cls=combine_classes(alert, alert_colors.error))
    
    # Build progress bars dynamically
    bars = []
    for bar_id, bar in snapshot['bars'].items():
        bars.append(
            Div(
                P(f"{bar.description}: {bar.progress:.1f}% ({bar.current}/{bar.total})",
                  cls=str(font_size.sm)),
                Progress(
                    value=str(bar.progress),
                    max="100",
                    cls=combine_classes(progress, progress_colors.accent, w.full)
                ),
                cls=str(m.b(3))
            )
        )
    
    content = Div(
        # Job info
        Div(
            P(f"ID: {job_id}", cls=combine_classes(font_size.xs, text_color.gray(500))),
            P(f"Name: {meta.get('name', 'Unknown')}", cls=str(font_weight.semibold)),
            P(f"Type: {meta.get('type', 'unknown').title()}"),
            P(f"Created: {meta.get('created_at', 'Unknown')}", cls=str(font_size.sm)),
            cls=str(m.b(4))
        ),
        
        # Overall progress
        Div(
            P(f"Overall Progress: {snapshot['overall_progress']:.1f}%",
              cls=combine_classes(font_weight.bold, m.b(2))),
            Progress(
                value=str(snapshot['overall_progress']),
                max="100",
                cls=combine_classes(progress, progress_colors.primary, w.full, m.b(4))
            ),
        ),
        
        # Individual bars
        Div(
            H4("Task Progress:", cls=combine_classes(font_weight.semibold, m.b(2))),
            *bars
        ) if bars else "",
        
        # Results if available
        Div(
            H4("Results:", cls=combine_classes(font_weight.semibold, m.b(2))),
            Pre(
                json.dumps(result, indent=2),
                cls=combine_classes(
                    bg_dui.base_300, p(3), rounded(),
                    font_size.xs, overflow.auto, max_h(40)
                )
            ),
            cls=str(m.t(4))
        ) if result else "",
        
        # History if available
        Div(
            H4("History:", cls=combine_classes(font_weight.semibold, m.b(2))),
            P(f"{len(snapshot.get('history', []))} updates recorded", cls=str(font_size.sm)),
            cls=str(m.t(4))
        ) if snapshot.get('history') else ""
    )
    
    # Add auto-refresh if job is still running
    if not snapshot['completed']:
        content.attrs['hx-get'] = f"/job_details?job_id={job_id}"
        content.attrs['hx-trigger'] = "load delay:500ms"
        content.attrs['hx-swap'] = "outerHTML"
    
    return content

In [11]:
@rt("/cancel_job")
def cancel_job(job_id: str):
    success = runner.cancel(job_id)
    # Return updated queue
    return queue()

In [12]:
# Optional SSE endpoint for direct streaming if needed
@rt("/stream")
def stream(job_id: str):
    """SSE endpoint - available for direct EventSource usage if needed"""
    return EventStream(
        sse_stream(
            monitor,
            job_id,
            interval=0.05,
            heartbeat=15.0,
            wait_for_start=True,
            start_timeout=10.0
        )
    )

In [13]:
@rt("/clear_completed")
def clear_completed():
    # Get all jobs before clearing
    all_jobs = monitor.all()
    
    # Clear completed jobs from monitor
    monitor.clear_completed(older_than_seconds=0)
    
    # Also manually remove cancelled jobs (in case monitor didn't clear them)
    for job_id in list(all_jobs.keys()):
        meta = job_metadata.get(job_id, {})
        job = all_jobs.get(job_id, {})
        
        # Remove if completed or cancelled
        if job.get('completed') or meta.get('status') == 'cancelled':
            # Remove from monitor if still there
            if job_id in monitor._jobs:
                del monitor._jobs[job_id]
            
            # Clean up metadata
            if job_id in job_metadata:
                del job_metadata[job_id]
            
            # Clean up results
            if job_id in job_results:
                del job_results[job_id]
            
            # Remove from active jobs
            active_jobs.discard(job_id)
    
    # Return updated queue
    return queue()

In [16]:
# Start server for Jupyter
from cjm_fasthtml_daisyui.core.testing import start_test_server
from fasthtml.jupyter import HTMX
from IPython.display import display

server = start_test_server(app)
display(HTMX())

[STATS] Called
[STATS] Total=0, Running=0, Completed=0, Cancelled=0
[STATS] No polling needed - no running jobs
[STATS] Returning with attrs: {'id': 'stats', 'class': '/stats shadow w-full'}
[QUEUE] Called, last_button_state=False
[QUEUE] Found 0 jobs
[STATS] Called
[STATS] Total=0, Running=0, Completed=0, Cancelled=0
[STATS] No polling needed - no running jobs
[STATS] Returning with attrs: {'id': 'stats', 'class': '/stats shadow w-full'}
[QUEUE] Called, last_button_state=False
[QUEUE] Found 0 jobs
[CREATE_JOB] Starting job creation
[CREATE_JOB] Created job 8abf7883... type=batch


Batch 1/3:  40%|██████████████████████▊                                  | 20/50 [00:00<00:00, 196.02it/s]

[CREATE_JOB] After delay, checking monitor...
[CREATE_JOB] Setting last_button_state from False to True
[CREATE_JOB] Calling queue() to get content
[QUEUE] Called, last_button_state=True
[QUEUE] Found 1 jobs
[QUEUE] Job 8abf7883... status=Running, progress=0.0%
[QUEUE] has_running=True
[QUEUE] Adding polling triggers to wrapper
[QUEUE] Returning wrapper without button update
[CREATE_JOB] Queue content type: <class 'fastcore.xml.FT'>
[CREATE_JOB] Queue content attrs: {'hx-get': '/queue', 'hx-trigger': 'load delay:1s', 'hx-swap': 'innerHTML', 'hx-target': '#job-queue', 'hx-on-htmx-after-swap': "htmx.trigger('#stats', 'refresh')"}
[CREATE_JOB] Forcing button disabled state
[CREATE_JOB] Calling stats() to get content
[STATS] Called
[STATS] Total=1, Running=1, Completed=0, Cancelled=0
[STATS] Adding polling triggers (running jobs found)
[STATS] Returning with attrs: {'id': 'stats', 'class': '/stats shadow w-full', 'hx-get': '/stats', 'hx-trigger': 'load delay:2s', 'hx-swap': 'outerHTML'}
[C

Batch 3/3: 100%|█████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 196.19it/s]


[QUEUE] Called, last_button_state=True
[QUEUE] Found 1 jobs
[QUEUE] Job 8abf7883... status=Complete, progress=100.0%
[QUEUE] has_running=False
[QUEUE] Button state changed from True to False
[QUEUE] No polling needed - all jobs complete
[QUEUE] Returning wrapper with button update
[STATS] Called
[STATS] Total=1, Running=0, Completed=1, Cancelled=0
[STATS] No polling needed - no running jobs
[STATS] Returning with attrs: {'id': 'stats', 'class': '/stats shadow w-full'}
[CREATE_JOB] Starting job creation
[CREATE_JOB] Created job 65d95cfc... type=export


...

[QUEUE] Called, last_button_state=True
[QUEUE] Found 3 jobs
[QUEUE] Job 8abf7883... status=Complete, progress=100.0%
[QUEUE] Job 65d95cfc... status=Complete, progress=100.0%
[QUEUE] Job 9fda2b28... status=Running, progress=95.0%
[QUEUE] has_running=True
[QUEUE] Adding polling triggers to wrapper
[QUEUE] Returning wrapper without button update


Epoch 3/3 - Validation:  65%|███████████████████████████▉               | 130/200 [00:01<00:00, 98.32it/s]

[STATS] Called
[STATS] Total=3, Running=1, Completed=2, Cancelled=0
[STATS] Adding polling triggers (running jobs found)
[STATS] Returning with attrs: {'id': 'stats', 'class': '/stats shadow w-full', 'hx-get': '/stats', 'hx-trigger': 'load delay:2s', 'hx-swap': 'outerHTML'}
[QUEUE] Called, last_button_state=True
[QUEUE] Found 3 jobs
[QUEUE] Job 8abf7883... status=Complete, progress=100.0%
[QUEUE] Job 65d95cfc... status=Complete, progress=100.0%
[QUEUE] Job 9fda2b28... status=Running, progress=97.8%
[QUEUE] has_running=True
[QUEUE] Adding polling triggers to wrapper
[QUEUE] Returning wrapper without button update


Epoch 3/3 - Validation: 100%|███████████████████████████████████████████| 200/200 [00:02<00:00, 98.07it/s]


[QUEUE] Called, last_button_state=True
[QUEUE] Found 3 jobs
[QUEUE] Job 8abf7883... status=Complete, progress=100.0%
[QUEUE] Job 65d95cfc... status=Complete, progress=100.0%
[QUEUE] Job 9fda2b28... status=Complete, progress=100.0%
[QUEUE] has_running=False
[QUEUE] Button state changed from True to False
[QUEUE] No polling needed - all jobs complete
[QUEUE] Returning wrapper with button update
[STATS] Called
[STATS] Total=3, Running=0, Completed=3, Cancelled=0
[STATS] No polling needed - no running jobs
[STATS] Returning with attrs: {'id': 'stats', 'class': '/stats shadow w-full'}


In [17]:
# Stop server when done
server.stop()